# React → Leptos conversion eval

**Notebook version: 2026-11-20 v4** (if you don't see this line, reload from disk in Jupyter).

Convert TSX to Leptos 0.7 with a local Ollama model, verify with `cargo check`, grade fidelity, and compare models.

**Run the cells in order.** Cell 3 warms the cargo cache — slow the first time (a few minutes). Paths (`leptos-check/`, `out/`, `samples/`) are relative to the notebook's working directory.
Cell 7 (the prompt tester) is self-contained, so you can re-run it after a kernel restart without re-running everything else.


## Cell 1: Install

Python deps via pip. Rust/cargo is installed by [rustup](https://rustup.rs) in `~/.cargo/bin`;
this cell adds that directory to the kernel's `PATH`, then `!cargo --version` checks it.
If cargo is still missing, install Rust and restart the kernel.


In [1]:
%pip install -q gradio pandas

# rustup installs cargo in ~/.cargo/bin; a Jupyter kernel may not inherit that PATH.
import os, pathlib
cargo_bin = pathlib.Path.home() / ".cargo" / "bin"
if cargo_bin.is_dir():
    os.environ["PATH"] = f"{cargo_bin}{os.pathsep}{os.environ.get('PATH', '')}"

!cargo --version


Note: you may need to restart the kernel to use updated packages.
cargo 1.100.0 (3acc79c0c 2026-11-05)


## Cell 2: Config

`DEFAULT_MODEL` is the Ollama model the evaluator (final cell) uses; `CRATE` is the
scratch crate that generated code is written into for `cargo check`.


In [2]:
import os, re, shutil, subprocess, pathlib, json, time
import pandas as pd

# rustup installs cargo into ~/.cargo/bin, which can be missing from a Jupyter
# kernel's PATH even when it works in an interactive shell. Add it and resolve
# an absolute path once so every subprocess call below uses it.
CARGO_BIN_DIR = pathlib.Path.home() / ".cargo" / "bin"
if CARGO_BIN_DIR.is_dir() and str(CARGO_BIN_DIR) not in os.environ.get("PATH", "").split(os.pathsep):
    os.environ["PATH"] = f"{CARGO_BIN_DIR}{os.pathsep}{os.environ.get('PATH', '')}"
CARGO = shutil.which("cargo")
if CARGO is None:
    raise RuntimeError("cargo not found; install Rust via https://rustup.rs and restart the kernel")

DEFAULT_MODEL = "gpt-oss:20b"
CRATE = pathlib.Path("leptos-check").resolve()


## Cell 3: Scratch crate and compile check

`leptos-check` is a scratch lib crate the generated code gets written into for `cargo check`.

In [3]:
import re

ANSI_RE = re.compile(r"\x1b\[[0-9;]*m")

def _cargo_check(timeout=180):
    """Run cargo check on the scratch crate, returning (ok, clean stderr).

    --quiet drops cargo's "Checking ..." progress line so stderr starts with the
    first real diagnostic; ANSI codes are stripped so logs are readable in CSV.
    """
    r = subprocess.run(
        [CARGO, "check", "--quiet", "--message-format=short"],
        cwd=CRATE, capture_output=True, text=True, timeout=timeout,
        env={**os.environ, "RUSTFLAGS": "-D warnings"},
    )
    return r.returncode == 0, ANSI_RE.sub("", r.stderr).strip()

def bootstrap_crate():
    if not CRATE.exists():
        subprocess.run([CARGO, "new", "--lib", str(CRATE)], check=True)
        subprocess.run([CARGO, "add", "leptos@0.7", "--features", "csr"], cwd=CRATE, check=True)
        subprocess.run([CARGO, "add", "tailwind_fuse"], cwd=CRATE, check=True)
    # warm the cache so the first real check is fast
    (CRATE / "src" / "lib.rs").write_text("")
    return _cargo_check()

def compile_check(rust: str, timeout=180):
    (CRATE / "src" / "lib.rs").write_text(rust)
    return _cargo_check(timeout)

ok, log = bootstrap_crate()   # slow once, a few minutes
print("ready" if ok else log)


ready


## Cell 4: Static fidelity checks

Compiling doesn't prove fidelity, so these cover what `cargo check` can't see.

In [4]:
def fidelity_checks(tsx: str, rust: str) -> dict:
    c = {}
    c["uses_view_macro"] = "view!" in rust
    c["no_props_type"] = not re.search(r"\b(enum|struct)\s+\w*Props\b", rust)
    c["no_panics"] = not re.search(r"unreachable!|panic!|\.unwrap\(\)", rust)
    c["no_bad_import"] = not re.search(r"use\s+[\w:]+!\s*;", rust)
    c["helpers_module_level"] = True  # refine later if needed
    if re.search(r"children\?", tsx):
        c["optional_children"] = "Option<Children>" in rust
    if re.search(r"=\s*[\w'\"\d]+\s*[,}]", tsx):  # destructured defaults
        c["has_prop_default_or_optional"] = "#[prop(" in rust
    if "${" in tsx:  # dynamic tailwind classes in source
        c["dynamic_class_todo"] = "TODO" in rust
    return c

## Cell 5: Gradio UI

Verification harness — another agent produces the Leptos conversion. Paste the generated
Rust here and it runs `cargo check` plus the static fidelity checks (the TSX-conditional
checks are skipped without the React source). `share=False`; `server_name` defaults to localhost.


In [5]:
import gradio as gr

# Some gradio versions (e.g. 6.x) have no "rust" mode in gr.Code; fall back to plain text.
_RUST_LANG = "rust" if "rust" in gr.Code.languages else None

def ui_check(rust: str):
    if not rust.strip():
        return "paste a component first", {}
    ok, log = compile_check(rust)
    checks = fidelity_checks("", rust)  # no TSX source: source-conditional checks are skipped
    status = "compiled" if ok else "failed"
    return status + ("\n\n" + log if log else ""), checks

with gr.Blocks(title="Leptos checker") as demo:
    rust_in = gr.Code(language=_RUST_LANG, label="Leptos output", lines=25)
    status = gr.Textbox(label="Compile status / compiler output", lines=8)
    checks = gr.JSON(label="Fidelity checks")
    gr.Button("Check").click(ui_check, rust_in, [status, checks])

demo.launch(inline=True)   # use share=False; server_name defaults to localhost


* Running on local URL:  http://127.0.0.1:7864
* To create a public link, set `share=True` in `launch()`.


## Cell 6: Batch run, grade, compare

Grade **already-converted** Leptos files — another agent does the conversion. Put `.rs`
files in `converted/` (model identity usually already sits in the filename). A sibling
`.tsx` of the same stem is read when present so the source-conditional fidelity checks
fire. Each run writes `out/results.csv` with one row per file.


In [6]:
def run_batch(rs_dir="converted"):
    """Grade converted Leptos files: cargo check + fidelity checks per .rs file."""
    files = sorted(pathlib.Path(rs_dir).glob("**/*.rs"))
    if not files:
        raise FileNotFoundError(
            f"no .rs files found in {pathlib.Path(rs_dir).resolve()} "
            f"(cwd: {pathlib.Path.cwd()}) -- add converted files or pass rs_dir=..."
        )
    log_dir = pathlib.Path("out", "logs")
    log_dir.mkdir(parents=True, exist_ok=True)
    rows = []
    for f in files:
        rust = f.read_text()
        tsx_path = f.with_suffix(".tsx")
        tsx = tsx_path.read_text() if tsx_path.exists() else ""
        t0 = time.time()
        ok, log = compile_check(rust)
        (log_dir / f"{f.stem}.log").write_text("cargo check passed\n" if ok else log + "\n")
        checks = fidelity_checks(tsx, rust)
        rows.append({
            "file": f.name,
            "status": "compiled" if ok else "failed",
            "seconds": round(time.time() - t0, 1),
            "checks_passed": sum(checks.values()), "checks_total": len(checks),
            "first_error": next((l for l in log.splitlines() if "error" in l), ""),
        })
    df = pd.DataFrame(rows)
    out_dir = pathlib.Path("out")
    out_dir.mkdir(parents=True, exist_ok=True)
    df.to_csv(out_dir / "results.csv", index=False)
    return df

df = run_batch()
df


,file,status,seconds,checks_passed,checks_total,first_error
0,component-5a9r2b2e.rs,compiled,0.1,5,5,
1,component-mistral-latest-20261120-125513.rs,failed,0.2,3,5,src/lib.rs:4:45: error[E0665]: `#[derive(Defau...
2,component-mistral-latest-20261120-130748.rs,failed,0.1,2,5,"src/lib.rs:2:28: error: expected one of `::`, ..."
3,component-mistral-latest-20261120-131729.rs,failed,0.1,3,5,"src/lib.rs:2:28: error: expected one of `::`, ..."


## Cell 7: Evaluation prompt tester

Pick a model from `models.toml`, pick a converted component, optionally edit the system prompt
in the UI, and Run. `cargo check` runs first and is sent as `<compiler_output>` (ground truth);
a sibling `.tsx` is sent as `<react_source>`. Reports go to `out/evaluations/<file>.md`.

Self-contained: after a kernel restart you can run only this cell. It uses `leptos-check/`
(created by Cell 3) and finds `models.toml` automatically. Needs Ollama running with the
selected model pulled.


In [1]:
# Cell 7 marker v4  (2026-11-20) -- if a traceback lands on evaluate_component
# before DEFAULT_MODEL exists, Jupyter is running a stale buffer: reload from disk.
import os, re, pathlib, json, time
import urllib.request, tomllib
import gradio as gr

DEFAULT_MODEL = globals().get("DEFAULT_MODEL") or "mistral:latest"
print("Cell 7 v4 loaded -- standalone prompt tester")

# Self-contained cargo check: Cell 7 works after a kernel restart on its own.
if "compile_check" not in globals():
    import shutil, subprocess

    _CARGO = shutil.which("cargo") or str(pathlib.Path.home() / ".cargo" / "bin" / "cargo")
    _CRATE = pathlib.Path("leptos-check").resolve()
    _ANSI = re.compile(r"\x1b\[[0-9;]*m")

    def compile_check(rust: str, timeout: int = 180):
        if not _CRATE.exists():
            raise RuntimeError("leptos-check/ is missing; run Cell 3 once, then retry")
        (_CRATE / "src" / "lib.rs").write_text(rust)
        r = subprocess.run(
            [_CARGO, "check", "--quiet", "--message-format=short"],
            cwd=_CRATE, capture_output=True, text=True, timeout=timeout,
            env={**os.environ, "RUSTFLAGS": "-D warnings"},
        )
        return r.returncode == 0, _ANSI.sub("", r.stderr).strip()


EVAL_PROMPT = """You are a senior Rust and Leptos engineer acting as a strict, evidence-based code reviewer. You evaluate a Leptos 0.7 component that was automatically converted from a React (TSX) component. Your job is to grade the conversion and explain the grade.

## Inputs
You will receive:
1. <react_source>: the original React/TypeScript component (and any helpers it uses, e.g. cn(), types).
2. <leptos_output>: the converted Leptos (Rust) component.
3. Optionally <compiler_output>: the result of `cargo check` / `cargo leptos build`, and <test_output>.
If compiler output is provided, treat it as ground truth. If it is not provided, say so, and do not claim the code compiles or fails to compile. Judge by reading and mark compile-related findings as "unverified".

## Rules of evaluation
- Be evidence-based. Every deduction must cite a specific line, snippet, or construct from the input. No generic criticism.
- Do not reward length, comments, or confident-looking code. Reward correctness and fidelity.
- Do not penalise valid idiomatic differences (e.g. signals instead of useState, `children: Children` instead of `children` prop).
- Target is Leptos 0.7 (reactive_graph, `#[component]`, `view!` macro, `signal()` / `RwSignal`, `Effect::new`, `Memo::new`, `NodeRef`, `Callback`, `ChildrenFn`, `Show`, `For`). Flag any pre-0.7 or invented APIs (e.g. `create_signal`, `cx: Scope`, `create_effect`, `use_state`) as errors.
- Flag hallucinated crates, functions, props, or attributes explicitly.
- If the React source is ambiguous, say so rather than guessing intent.
- Score each criterion independently before computing the total.

## Rubric (100 points)

1. Compilability and Leptos 0.7 API correctness (25)
   - 22-25: Compiles (or would clearly compile); only valid 0.7 APIs; correct imports; correct types and lifetimes/ownership; no hallucinated APIs.
   - 15-21: Minor issues (a missing import, small type mismatch) fixable in under 5 minutes.
   - 8-14: Several errors, or one structural error (wrong signal/closure/ownership model).
   - 0-7: Does not compile in a way that needs a rewrite, or uses the wrong framework version throughout.

2. Behavioural / functional fidelity (25)
   - Same rendered output and behaviour as the React source for the same inputs: conditionals, lists, event handlers, state transitions, effects, refs, and context.
   - 22-25: All behaviour preserved. 15-21: Minor behavioural drift. 8-14: Notable missing or wrong behaviour. 0-7: Largely different or non-functional.

3. Props, types and API surface (15)
   - TS props mapped to Rust props correctly: optional props (`#[prop(optional)]`, `#[prop(into)]`, `#[prop(default = ...)]`), children, callbacks, generics, enums for string unions, correct Rust types.
   - Component signature is usable by a caller the same way as the original.

4. Reactivity and state model (15)
   - Correct use of signals, memos and effects; no needless clones or stale captures; derived values as closures/Memo; no effects used where derived state belongs; dependency semantics preserved (React deps arrays mapped to tracked reads).
   - Closures invoked where reactive reads are needed (e.g. `move || ...` in `view!`).

5. Markup, styling and accessibility fidelity (10)
   - Elements, attributes, class names/Tailwind classes, conditional classes (cn()/clsx equivalent), `class:` directives, aria-* and role attributes, `for`/`id`, event names, `style` handling, and DOM structure preserved.

6. Idiomatic Rust / Leptos quality (5)
   - Clear structure, appropriate use of `Show`/`For`/`Suspense`, no unnecessary `unwrap()`, no needless `clone()`, sensible module and naming conventions, no dead code.

7. Completeness and honesty (5)
   - Nothing silently dropped (features, props, edge cases). Unsupported parts are marked clearly with TODOs rather than faked. No fabricated helper functions that are never defined.

## Severity levels for findings
- critical: breaks compilation or core behaviour
- major: wrong behaviour in a realistic case, or wrong API surface
- minor: small drift, style, or unverified concern
- nit: cosmetic

## Grade bands
- 90-100 A: Production-ready with at most trivial edits
- 75-89 B: Good; small fixes needed
- 60-74 C: Usable skeleton; meaningful fixes needed
- 40-59 D: Major rework needed
- 0-39 F: Not useful

## Process
1. Inventory the React component: props, state, effects, refs, context, event handlers, conditional/list rendering, styling helpers, and external dependencies.
2. Build a mapping table from each React construct to its Leptos counterpart, and mark each as Correct / Partial / Missing / Wrong.
3. Review the Leptos code against the rubric, collecting findings with evidence.
4. Score each criterion, sum them, and apply the grade band.
5. Cap rule: if the code does not compile (confirmed by compiler output) the total cannot exceed 59; if it uses a wrong framework version throughout, the total cannot exceed 39.
6. Suggest fixes, ordered by impact.

## Output format
Return exactly two parts.

Part 1: a short human-readable report in Markdown:
- Verdict (one sentence), total score, letter grade
- Construct mapping table (React construct | Leptos equivalent | Status)
- Scores table (criterion | score/max | one-line justification)
- Top findings (severity, location, evidence, suggested fix)
- Minimal corrected snippet for the most important critical/major issue, if any

Part 2: a single JSON object in a ```json block, with this exact shape:
{
  "total": <int 0-100>,
  "grade": "A|B|C|D|F",
  "compile_verified": <true|false>,
  "scores": {
    "compilability_api": {"score": <int>, "max": 25},
    "behavioural_fidelity": {"score": <int>, "max": 25},
    "props_types": {"score": <int>, "max": 15},
    "reactivity_state": {"score": <int>, "max": 15},
    "markup_styling_a11y": {"score": <int>, "max": 10},
    "idiomatic_quality": {"score": <int>, "max": 5},
    "completeness_honesty": {"score": <int>, "max": 5}
  },
  "findings": [
    {"severity": "critical|major|minor|nit", "criterion": "<key>", "location": "<line or snippet>", "issue": "<text>", "fix": "<text>"}
  ],
  "hallucinated_apis": ["<name>", ...],
  "summary": "<2 sentences>"
}
The scores must sum to `total`. Output nothing after the JSON."""
def build_eval_input(rust: str, tsx: str = "", compile_log: str = "") -> str:
    parts = [
        f"<react_source>\n{tsx or '[not provided]'}\n</react_source>",
        f"<leptos_output>\n{rust}\n</leptos_output>",
    ]
    if compile_log:
        parts.append(f"<compiler_output>\n{compile_log}\n</compiler_output>")
    return "\n\n".join(parts)

# Native /api/chat (not the OpenAI-compat /v1) so num_ctx actually applies:
# ollama defaults the context window to 4096, which clamps generation to
# 4096 - prompt_tokens and truncates long rubric reports before the JSON.
def evaluate_component(rust: str, tsx: str = "", compile_log: str = "", model: str = DEFAULT_MODEL,
                       num_ctx: int = 16384, max_tokens: int = 8192, temperature: float = 0.1,
                       ollama_url: str = "http://localhost:11434", retries: int = 1,
                       system_prompt: str = EVAL_PROMPT) -> dict:
    payload = {
        "model": model,
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": build_eval_input(rust, tsx, compile_log)},
        ],
        "stream": False,
        "options": {"num_ctx": num_ctx, "num_predict": max_tokens, "temperature": temperature},
    }
    raw = ""
    for _ in range(retries + 1):
        req = urllib.request.Request(
            f"{ollama_url}/api/chat",
            data=json.dumps(payload).encode(),
            headers={"Content-Type": "application/json"},
        )
        with urllib.request.urlopen(req, timeout=600) as resp:
            raw = json.load(resp)["message"]["content"] or ""
        m = re.search(r"```json\s*(\{.*\})\s*```", raw, re.S) or re.search(r"```json\s*(\{.*)", raw, re.S)  # fence opened but response cut before it closed
        if m:
            try:
                return {"raw": raw, "report": raw[: m.start()].strip(), "result": json.loads(m.group(1))}
            except json.JSONDecodeError:
                pass
    return {"raw": raw, "report": raw, "result": None}

# --- Inputs: models.toml + converted/ ---
def find_models_toml(start=None):
    start = pathlib.Path(start or pathlib.Path.cwd()).resolve()
    for d in (start, *start.parents):
        for rel in ("models.toml", "conversion/models.toml", "prompt-eval/models.toml"):
            cand = d / rel
            if cand.exists():
                return cand
    return None

MODELS_TOML = find_models_toml()
MODEL_CONFIG, OLLAMA_URL = {}, "http://localhost:11434"
if MODELS_TOML:
    with open(MODELS_TOML, "rb") as fh:
        _cfg = tomllib.load(fh)
    OLLAMA_URL = _cfg.get("ollama", {}).get("url", OLLAMA_URL)
    MODEL_CONFIG = _cfg.get("models", {})
print(f"models.toml: {MODELS_TOML or 'not found'} ({len(MODEL_CONFIG)} models)")

CONVERTED_DIR = pathlib.Path("converted")
EVAL_OUT = pathlib.Path("out", "evaluations")

def converted_files(root=CONVERTED_DIR):
    return [str(p) for p in sorted(root.glob("**/*.rs"))]

# --- One run: cargo check, then the (possibly edited) prompt, then the report ---
def run_prompt_test(model_name, rs_file, system_prompt, temperature, max_tokens):
    if not rs_file:
        return "Pick a converted file.", "", "", {}, ""
    path = pathlib.Path(rs_file)
    if not path.exists():
        return f"File not found: {path}", "", "", {}, ""

    rust = path.read_text()
    tsx_path = path.with_suffix(".tsx")
    tsx = tsx_path.read_text() if tsx_path.exists() else ""

    ok, log = compile_check(rust)
    compile_log = "cargo check passed (exit 0)" if ok else "\n".join(log.splitlines()[-80:])

    cfg = MODEL_CONFIG.get(model_name, {})
    t0 = time.time()
    ev = evaluate_component(
        rust, tsx=tsx, compile_log=compile_log,
        model=cfg.get("model_id", model_name or DEFAULT_MODEL),
        num_ctx=int(cfg.get("num_ctx", 16384)),
        max_tokens=int(max_tokens), temperature=float(temperature),
        ollama_url=OLLAMA_URL, system_prompt=system_prompt or EVAL_PROMPT,
    )
    elapsed = round(time.time() - t0, 1)

    EVAL_OUT.mkdir(parents=True, exist_ok=True)
    (EVAL_OUT / f"{path.stem}.md").write_text(ev["raw"])

    r = ev["result"] or {}
    status = (
        f"**{path.name}**  \n"
        f"model: `{cfg.get('model_id', model_name or DEFAULT_MODEL)}` · {elapsed}s · "
        f"prompt: {len(system_prompt or EVAL_PROMPT)} chars  \n"
        f"cargo: **{'passed' if ok else 'FAILED'}** · "
        f"grade: **{r.get('grade', 'unparsed')}** ({r.get('total', '?')}/100)"
    )
    return status, compile_log, ev["report"] or ev["raw"], r, ev["raw"]

_mod_choices = list(MODEL_CONFIG) or [DEFAULT_MODEL]
_mod_default = next(
    (k for k in _mod_choices if DEFAULT_MODEL in (k, MODEL_CONFIG.get(k, {}).get("model_id"))),
    _mod_choices[0],
)
_files = converted_files()

with gr.Blocks(title="Leptos prompt tester") as eval_demo:
    gr.Markdown(
        f"### Evaluation prompt tester\n"
        f"Pick a model and a converted component, edit the prompt if you need to, then Run. "
        f"`cargo check` runs first and is sent as `<compiler_output>` (ground truth). "
        f"Reports are saved to `{EVAL_OUT}/<file>.md`."
    )
    with gr.Row():
        with gr.Column(scale=1):
            model_dd = gr.Dropdown(choices=_mod_choices, value=_mod_default, label="Model")
            file_dd = gr.Dropdown(choices=_files, value=_files[0] if _files else None, label="Converted file")
            with gr.Row():
                temp_sl = gr.Slider(0.0, 1.0, value=0.1, step=0.05, label="Temperature")
                max_tok = gr.Slider(1024, 16384, value=8192, step=1024, label="Max tokens")
            with gr.Accordion("System prompt (editable)", open=False):
                prompt_tb = gr.Textbox(value=EVAL_PROMPT, lines=18, show_label=False)
                reset_btn = gr.Button("Reset to default", size="sm")
            with gr.Row():
                run_btn = gr.Button("Run test", variant="primary")
                refresh_btn = gr.Button("Refresh files", size="sm")
            status_md = gr.Markdown()
            with gr.Accordion("Compiler output", open=False):
                compile_tb = gr.Textbox(lines=10, interactive=False, show_label=False)
        with gr.Column(scale=3):
            with gr.Tabs():
                with gr.Tab("Report"):
                    report_md = gr.Markdown()
                with gr.Tab("JSON"):
                    result_json = gr.JSON()
                with gr.Tab("Raw"):
                    raw_tb = gr.Textbox(lines=25, interactive=False, show_label=False)

    run_btn.click(
        run_prompt_test,
        inputs=[model_dd, file_dd, prompt_tb, temp_sl, max_tok],
        outputs=[status_md, compile_tb, report_md, result_json, raw_tb],
    )
    reset_btn.click(lambda: EVAL_PROMPT, outputs=prompt_tb)
    refresh_btn.click(lambda: gr.Dropdown(choices=converted_files()), outputs=file_dd)
    eval_demo.queue()

eval_demo.launch(inline=True)  # add share=True for a public link


Cell 7 v4 loaded -- standalone prompt tester
models.toml: /home/daniel/Documents/ai-project/AI-AGENTS/react-to-leptos/conversion/models.toml (8 models)
* Running on local URL:  http://127.0.0.1:7864
* To create a public link, set `share=True` in `launch()`.
